In [ ]:
# TQSdk 学习示例代码 - 带回测功能的双均线策略
from datetime import date
from tqsdk import TqApi, TqAuth, TqBacktest, TargetPosTask
from tqsdk.exceptions import BacktestFinished

def main():
    # 1. 初始化API连接 (使用回测模式)
    api = TqApi(
        backtest=TqBacktest(
            start_dt=date(2023, 10, 1),  # 回测开始日期
            end_dt=date(2023, 12, 31)   # 回测结束日期
        ),
        web_gui=True,  # 开启图形化界面
        auth=TqAuth("QiZhihui", "QZH123"),  # 您的账号
    )
    
    try:
        # 2. 获取螺纹钢主力合约行情
        symbol = "SHFE.rb2401"  # 合约代码可能需要根据实际情况调整
        quote = api.get_quote(symbol)
        print(f"获取{symbol}行情数据...")

        # 3. 获取K线数据 (15分钟线)
        klines = api.get_kline_serial(symbol, 15*60, data_length=50)
        print(f"已获取{len(klines)}根K线数据")

        # 4. 创建目标持仓任务
        target_pos = TargetPosTask(api, symbol)

        # 初始化变量记录上一次的均线状态
        last_ma5 = None
        last_ma20 = None

        # 5. 改进版双均线策略
        try:
            while True:
                api.wait_update()
                if api.is_changing(klines):
                    # 计算5日和20日均线
                    ma5 = klines.close.iloc[-5:].mean()
                    ma20 = klines.close.iloc[-20:].mean()
                    
                    # 交易信号判断（只在真正的金叉/死叉时触发）
                    if last_ma5 is not None and last_ma20 is not None:
                        # 金叉条件：前一根K线MA5 <= MA20，当前K线MA5 > MA20
                        if last_ma5 <= last_ma20 and ma5 > ma20:
                            print(f"最新价: {quote.last_price}: 金叉信号: MA5({ma5:.2f})上穿MA20({ma20:.2f})，做多1手")
                            target_pos.set_target_volume(1)
                            
                        
                        # 死叉条件：前一根K线MA5 >= MA20，当前K线MA5 < MA20
                        elif last_ma5 >= last_ma20 and ma5 < ma20:
                            print(f"最新价: {quote.last_price}: 死叉信号: MA5({ma5:.2f})下穿MA20({ma20:.2f})，做空1手")
                            target_pos.set_target_volume(-1)
                            
                    
                    # 保存当前均线值供下次比较
                    last_ma5 = ma5
                    last_ma20 = ma20
                            
        except BacktestFinished:
            print("---------- 回测正常结束 ----------")

    finally:
        # 6. 关闭API连接
        api.close()
        print("API连接已关闭")

if __name__ == "__main__":
    main()


Hello world!


In [1]:


'''
Aberration策略 (难度：初级)
参考: https://www.shinnytech.com/blog/aberration/
注: 该示例策略仅用于功能示范, 实盘时请根据自己的策略/经验进行修改
'''

#from tqsdk import TqApi, TqAuth, TargetPosTask
from tqsdk.ta import BOLL
from datetime import date
from tqsdk import TqApi, TqAuth, TqBacktest, TargetPosTask
from tqsdk.exceptions import BacktestFinished

# 设置合约代码
SYMBOL = "DCE.i2001"
api = TqApi(
    backtest=TqBacktest(
        start_dt=date(2019, 4, 19),
        end_dt=date(2019, 8, 26)
    ),
    web_gui=True,  # 开启图形化界面
    auth=TqAuth("QiZhihui", "QZH123")
)
quote = api.get_quote(SYMBOL)
klines = api.get_kline_serial(SYMBOL, 60 * 60)
position = api.get_position(SYMBOL)
target_pos = TargetPosTask(api, SYMBOL)


# 使用BOLL指标计算中轨、上轨和下轨，其中27为周期N，2为参数p
def boll_line(klines):
    boll = BOLL(klines, 27, 2)
    midline = boll["mid"].iloc[-1]
    topline = boll["top"].iloc[-1]
    bottomline = boll["bottom"].iloc[-1]
    #print("策略运行，中轨：%.2f，上轨为:%.2f，下轨为:%.2f" % (midline, topline, bottomline))
    return midline, topline, bottomline


midline, topline, bottomline = boll_line(klines)

try:
    while True:
        api.wait_update()
        # 每次生成新的K线时重新计算BOLL指标
        if api.is_changing(klines.iloc[-1], "datetime"):
            midline, topline, bottomline = boll_line(klines)

        # 每次最新价发生变化时进行判断
        if api.is_changing(quote, "last_price"):
            # 判断开仓条件
            if position.pos_long == 0 and position.pos_short == 0:
                # 如果最新价大于上轨，K线上穿上轨，开多仓
                if quote.last_price > topline:
                    print("K线上穿上轨，开多仓")
                    target_pos.set_target_volume(200)
                # 如果最新价小于轨，K线下穿下轨，开空仓
                elif quote.last_price < bottomline:
                    print("K线下穿下轨，开空仓")
                    target_pos.set_target_volume(-200)
                else:
                    print("当前最新价%.2f,未穿上轨或下轨，不开仓" % quote.last_price)

            # 在多头情况下，空仓条件
            elif position.pos_long > 0:
                # 如果最新价低于中线，多头清仓离场
                if quote.last_price < midline:
                    print("最新价低于中线，多头清仓离场")
                    target_pos.set_target_volume(0)
                else:
                    print("当前多仓，未穿越中线，仓位无变化")

            # 在空头情况下，空仓条件
            elif position.pos_short > 0:
                # 如果最新价高于中线，空头清仓离场
                if quote.last_price > midline:
                    print("最新价高于中线，空头清仓离场")
                    target_pos.set_target_volume(0)
                else:
                    print("当前空仓，未穿越中线，仓位无变化")

except BacktestFinished:
    api.close()
    print("---------- 回测正常结束 ----------")

在使用天勤量化之前，默认您已经知晓并同意以下免责条款，如果不同意请立即停止使用：https://www.shinnytech.com/blog/disclaimer/


    INFO - 您可以访问 http://127.0.0.1:49891 查看策略绘制出的 K 线图形。
当前最新价572.50,未穿上轨或下轨，不开仓
当前最新价570.00,未穿上轨或下轨，不开仓
当前最新价569.50,未穿上轨或下轨，不开仓
当前最新价570.00,未穿上轨或下轨，不开仓
当前最新价569.50,未穿上轨或下轨，不开仓
当前最新价571.00,未穿上轨或下轨，不开仓
当前最新价570.50,未穿上轨或下轨，不开仓
当前最新价570.00,未穿上轨或下轨，不开仓
当前最新价569.50,未穿上轨或下轨，不开仓
当前最新价570.00,未穿上轨或下轨，不开仓
当前最新价570.50,未穿上轨或下轨，不开仓
当前最新价570.00,未穿上轨或下轨，不开仓
当前最新价569.50,未穿上轨或下轨，不开仓
当前最新价570.00,未穿上轨或下轨，不开仓
当前最新价571.00,未穿上轨或下轨，不开仓
当前最新价572.00,未穿上轨或下轨，不开仓
当前最新价572.50,未穿上轨或下轨，不开仓
当前最新价571.50,未穿上轨或下轨，不开仓
当前最新价572.00,未穿上轨或下轨，不开仓
当前最新价572.50,未穿上轨或下轨，不开仓
当前最新价572.00,未穿上轨或下轨，不开仓
当前最新价571.50,未穿上轨或下轨，不开仓
当前最新价572.00,未穿上轨或下轨，不开仓
当前最新价572.50,未穿上轨或下轨，不开仓
当前最新价572.00,未穿上轨或下轨，不开仓
当前最新价573.00,未穿上轨或下轨，不开仓
当前最新价573.50,未穿上轨或下轨，不开仓
当前最新价573.00,未穿上轨或下轨，不开仓
当前最新价574.00,未穿上轨或下轨，不开仓
当前最新价574.50,未穿上轨或下轨，不开仓
当前最新价575.00,未穿上轨或下轨，不开仓
当前最新价574.50,未穿上轨或下轨，不开仓
当前最新价574.00,未穿上轨或下轨，不开仓
当前最新价573.50,未穿上轨或下轨，不开仓
当前最新价574.00,未穿上轨或下轨，不开仓
当前最新价573.50,未穿上轨或下轨，不开仓
当前最新价573.00,未穿上轨或下轨，不开仓
当前最新价574.00,未穿上轨或下轨，不开仓
当前最新价574.50,未穿上轨或下轨，不开仓
当前最新价574

KeyboardInterrupt: 